# AIFM Hedge Fund
**Risk reporting**

AIFM Risk limits are defined in the fund's offering document and monitored against
internal thresholds. No regulatory VaR limit applies (unlike UCITS).

Key regulatory obligations under AIFMD:
- **Leverage**: gross and commitment method (EU 231/2013 Annex II)
- **Stress testing**: market, liquidity, and counterparty scenarios (ESMA/2020/1498)
- **Liquidity risk**: portfolio liquidity profile and redemption stress
- **Regulatory reporting**: quarterly Annex IV to CSSF — AIFMD II (Directive 2024/927/EU) 
  expanded requirements adding liquidity management tools, loan origination, and delegation arrangements

| | AIFMD I | AIFMD II |
|---|---|---|
| **Directive** | 2011/61/EU | 2024/927/EU |
| **Luxembourg law** | Law of 12 July 2013 | Law of 3 March 2026 (Bill 8628) |
| **In force** | 22 July 2013 | 16 April 2026  (+1y enforced)|

*  **ESMA:** Annex IV field definitions v1.7 (July 2024) 
*  Liquidity stress testing ESMA/2020/1498

Example in this nb: AIFM Hedge Fund, Strategy: Long/short equity, bonds, FX, options.

In [ ]:
from src.config import VALUATION_DATE, QUARTER
from src.risk.reg_constants import CONFIDENCE, HORIZON, NOTICE, GROSS_LIMIT

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.setup_db import run as setup_db
setup_db()

from src.ui.plot_style import (
    C, ACCENT, ACCENT2, ACCENT3, WARNING, section_title, sup_title, FONT_FAMILY,
)
from src.data.database import get_engine, query_positions, query_nav_history
from src.data.enrichment import get_risk_ready_df
from src.data.mock_bloomberg import MockBloomberg as Bloomberg
from src.risk.leverage_config import INSTRUMENT_SOURCE
import src.risk.risk_utils as rk
import src.ui.print_html_utils as phtml
import src.print_utils as prt
import src.risk.esg_utils as esg_u 
from src.risk.esg_utils import ESG_FIELDS, ESG_THRESHOLD_LOW
from src.ui.nb_utils import save_fig, save_table, styled_table
from src.ui.print_html_utils import display_dark_table

FUND_ID = 'AIFM_HedgeFund'
ENGINE = get_engine()
BBG = Bloomberg()


---

## 1. Load and Validate Positions

Positions are queried from SQLite, which is loaded daily from the fund administrator
Excel export. The flow is:

1. Fund admin Excel → load_positions() → SQLite
2. SQLite → query_positions() → notebook

`get_risk_ready_df` runs the enrichment pipeline on the raw positions:
1. iquid instruments (equities, bonds, ETFs): 
- sensitivities fetched from Bloomberg
  (beta, modified duration, convexity, spread duration)
2 illiquid instruments (loans, direct properties): 
- fund admin data already embedded in the position file
- eg. rating, maturity, LTV, rental yield
- no Bloomberg ticker available or needed

The output is a single enriched DataFrame per fund per date, ready for VaR,
stress testing, and liquidity analysis.

In [ ]:
# query from SQLdb
positions = query_positions(ENGINE, FUND_ID, VALUATION_DATE)

# enrich w/ info from Bloomberg
risk_df = get_risk_ready_df(ENGINE, FUND_ID, VALUATION_DATE)

# info in risk_df covers many risk inputs
NAV = risk_df['market_value_eur'].sum()

phtml.display_fund_summary(FUND_ID, VALUATION_DATE, positions, risk_df, NAV)

In [ ]:
phtml.display_top_positions(risk_df)

In [ ]:
# Asset class breakdown
phtml.display_asset_class_breakdown(risk_df)  # Also computes NAV internally

In [ ]:
from src.ui.plot_utils import plot_breakdown_horizontal

plot_breakdown_horizontal(
    risk_df,
    title='Asset Class Breakdown',
    group_by='asset_class',
    fund_id=FUND_ID,
    filename="01. Asset Breakdown HF"
)


---

## 2. VaR and Expected Shortfall

Under AIFMD there is no regulatory VaR limit, but VaR is monitored against
internal limits defined in the RMP - Risk Management Policy - and reported in Annex IV.

- Confidence: 99%
- Horizon: 1-day and 20-day (square root of time scaling)
- Method: historical simulation, 250-day rolling window

> Note: Var methods (historic, parametric, montecarlo) choice
> might be used in conjunction as each has its pros and cons.

In [ ]:
nav_history = query_nav_history(ENGINE, FUND_ID)
nav_history['date'] = pd.to_datetime(nav_history['date'])
returns = nav_history['pnl_pct'].dropna().values

# historical simulation + 20d scaling
var_1d  = rk.var_historical(returns[-250:], confidence=CONFIDENCE)
var_20d = rk.var_scale(var_1d, horizon=HORIZON)
es_1d   = rk.es_historical(returns[-250:], confidence=CONFIDENCE)
es_20d  = rk.es_scale(es_1d, horizon=HORIZON)

phtml.display_var_es(var_1d, var_20d, es_1d, es_20d, NAV)


### 2.2 Liquidity-Adjusted VaR

Standard VaR assumes positions can be liquidated instantly at market price.
LVaR extends this by adding the estimated cost of unwinding positions under
stressed market conditions:

$$\text{LVaR} = \text{VaR} + \text{Liquidity Cost}$$

$$\text{Liquidity Cost}_i = \frac{1}{2} \times \text{spread}_i \times \text{stress multiplier}_i \times |MV_i|$$

LVaR is not a regulatory requirement. It originates from banking regulation
(Basel III internal models) and academic literature (Amihud & Mendelson, BIS
working papers) and is used internally by risk managers to capture the
liquidity dimension of market risk.

Spreads and stress multipliers are illustrative values adopted in this
notebook. In practice these are internal RMP parameters calibrated by the
fund manager and reviewed periodically.
<small> 
| Asset Class      | Normal Spread | Stress Multiplier | Stressed Spread       |
|------------------|---------------|-------------------|-----------------------|
| Large cap equity | 5bps          | 3x                | 15bps                 |
| Small cap equity | 20bps         | 5x                | 100bps                |
| IG bonds         | 10bps         | 5x                | 50bps                 |
| HY bonds         | 50bps         | 10x               | 500bps                |
| Senior loans     | 100bps        | 20x               | 2000bps               |
| Listed REITs     | 15bps         | 5x                | 75bps                 |
| Direct property  | N/A           | N/A               | 5-8% transaction cost |

</small> 

Mandatory liquidity risk management (buckets, redemption stress, LMTs) is
covered in Section 5.

In [ ]:
# MRS-33: Liquidity-adjusted VaR
# the function implemented here gets a fix stress multiplier
# RMP multiplier is a floor; risk committee may apply add-ons
# during stressed market conditions
lvar_result = rk.liquidity_adjusted_var(var_1d, risk_df, stress_multiplier=5)

phtml.display_lvar(lvar_result, NAV)

---

## 3. VaR Backtest (Kupiec + Christoffersen + ESMA)

VaR backtesting compares predicted VaR against realised daily P&L.
Two statistical tests are applied:

- **Kupiec POF**: tests whether the breach rate equals the expected rate (1% for 99% VaR)
- **Christoffersen**: tests whether breaches are independently distributed over time

> **Note**: Kupiec and Christoffersen tests are industry best practice and not explicitly
> required by ESMA or CSSF. The regulatory requirement is the 250-day breach count
> and zone classification only. Both tests should be documented in the RMP. For both
> models p > 0.05: fail to reject the null, model is statistically acceptable.

> **Private debt note**: NAV returns reflect mark-to-model valuations from the fund
> administrator, which smooth daily volatility. Breach counts will likely be lower
> than for a liquid fund. Results should be interpreted with this in mind.

ESMA (CESR/10-788) requires backtesting 1-day 99% VaR against realised daily P&L
over a 250 trading day rolling window.

Zone classification (Basel traffic light, adopted by ESMA):
- **Green** (0-4 breaches): model acceptable
- **Amber** (5-9 breaches): explanation required, possible model review
- **Red** (10+ breaches): model must be revised, CSSF notification required

#### 3.1. Internal (maximum history)

In [ ]:
# MRS-27: VaR backtest
nav_history = query_nav_history(ENGINE, FUND_ID)
nav_history['date'] = pd.to_datetime(nav_history['date'])

returns = nav_history['pnl_pct'].dropna().values
dates = nav_history['date'].iloc[1:].reset_index(drop=True)

# first VaR observation is at index 250, aligned with returns and dates
# the first 250 returns are used to compute the first VaR observation...
window = 250
var_hist = pd.Series([
    rk.var_historical(returns[max(0, i-window):i], confidence=0.99)
    for i in range(window, len(returns))
])

returns_aligned = returns[window:]
dates_aligned = dates.iloc[window:].reset_index(drop=True)

print(f"Backtest observations : {len(returns_aligned)}")

In [ ]:
report = rk.full_backtest_report(
    returns_series=pd.Series(returns_aligned),
    var_dict={'Historical': var_hist},
    dates=dates_aligned
)

window_size = len(returns_aligned)
phtml.display_backtest_report(report, window_size)


In [ ]:
from src.ui.backtest_plot import plot_var_backtest
plot_var_backtest(dates_aligned, returns_aligned, var_hist, FUND_ID, report_type='full')


#### 3.2. ESMA (250 trading days)

In [ ]:
returns_250 = returns_aligned[-250:]
var_250 = var_hist.iloc[-250:]
esma_250  = rk.exception_report(pd.Series(returns_250),
                              var_250, confidence=0.99, verbose=False)
n_250     = len(esma_250)
breach_250 = n_250 / 250
zone_250  = 'Green' if n_250 <= 4 else 'Amber' if n_250 <= 9 else 'Red'

phtml.display_esma_report(n_250, breach_250, zone_250)


In [ ]:
plot_var_backtest(dates_aligned, returns_aligned, var_hist, FUND_ID, report_type='250d', zone=zone_250)

In [ ]:
report = rk.full_backtest_report(
    returns_series=returns_250,
    var_dict={'Historical': var_250},
    dates=dates_aligned
)

phtml.display_backtest_report(report)

---

## 4. Leverage (Annex IV)

AIFMD requires leverage to be reported using two methods:

- **Gross method**: sum of absolute exposures divided by NAV. No netting allowed.
  Derivatives converted to equivalent underlying exposure.
- **Commitment method**: hedging and netting arrangements are recognised.
  Offsetting positions in the same underlying reduce exposure.

Limits are set in the fund's offering document and reported quarterly to the CSSF
in Annex IV. AIFMD II (Directive 2024/927/EU) expanded reporting requirements,
including the breakdown by:
* asset class
* instrument type
* source: 
  - financial borrowing
  - synthetic leverage through derivatives
  - repo/reverse repo

The expanded disclosure makes it easier for regulators to identify funds building leverage through
derivatives rather than borrowing.

In [ ]:
from src.risk.leverage_computation import compute_leverage, build_bbg_maps

BORROWINGS_EUR = 0.0

lev = compute_leverage(risk_df, NAV, BBG, FUND_ID, borrowings_eur=BORROWINGS_EUR)

# Extract maps for pre-trade checks
currency_bbg_map, deriv_bbg_map = build_bbg_maps(FUND_ID)

gross_exposure = lev['gross_exposure']
gross_leverage = lev['gross_leverage']
commitment_exposure = lev['commitment_exposure']
commitment_leverage = lev['commitment_leverage']
deriv_notional_commitment = lev['deriv_notional_commitment']

risk_df = lev['risk_df']


In [ ]:
# AIFMD II granular leverage breakdown
granular = risk_df.groupby(['asset_class', 'sub_asset_class']).agg(
    gross_eur=('gross_exposure', 'sum'),
    n_positions=('isin', 'count')
    ).reset_index()

granular['gross_x_nav'] = granular['gross_eur'] / NAV
granular['source']      = granular.apply(
    lambda r: INSTRUMENT_SOURCE.get(
        (r['asset_class'], r['sub_asset_class']), ('Other', 'Other'))[0], axis=1)
granular['listed_otc']  = granular.apply(
    lambda r: INSTRUMENT_SOURCE.get(
        (r['asset_class'], r['sub_asset_class']), ('Other', 'Other'))[1], axis=1)

# exclude rows with zero gross exposure (cash)
granular = granular[granular['gross_eur'] > 0].sort_values('gross_eur', ascending=False)

# borrowings row — included per Recital 13; appended separately as it is not
# a position in risk_df but a funding liability
if BORROWINGS_EUR > 0:
    import pandas as pd
    borrow_row = pd.DataFrame([{
        'asset_class'   : 'Borrowing',
        'sub_asset_class': 'PB Financing',
        'gross_eur'     : BORROWINGS_EUR,
        'n_positions'   : 1,
        'gross_x_nav'   : BORROWINGS_EUR / NAV,
        'source'        : 'Financial Borrowing',
        'listed_otc'    : 'N/A',
    }])
    granular = pd.concat([granular, borrow_row], ignore_index=True)

In [ ]:
phtml.display_granular(granular, NAV)

---

## 5. Stress Testing (Annex VI)
AIFMD Annex VI requires AIFMs to conduct regular stress tests covering market, liquidity, 
and counterparty risk. Scenarios must be documented in the RMP, reviewed at least annually, 
and results reported to the CSSF via the Annex IV filing.

Unlike UCITS, AIFMD does not prescribe specific stress scenarios. The scenarios below are 
designed to reflect the material risk factors of a long/short equity fund with fixed income, 
FX, and derivative exposure. Historical scenarios — aligned with those prescribed under UCITS 
guidelines — are included alongside hypothetical shocks as a matter of good risk management 
practice and to facilitate comparability across fund types.

In production, AIFMs typically delegate 
full revaluation stress testing to third-party risk systems  (Bloomberg PORT, Axioma, or 
BlackRock Aladdin) which capture non-linear effects such as option gamma and interest rate 
convexity. For simplicity here we use a stress methodology that applies a first-order sensitivity approximation (delta-normal). This could be reasonably used for internal monitoring purposes.

$$\Delta P_i = \text{sensitivity}_i \times \text{shock}_i \times MV_i$$

Scenarios covered:
- **Equity crash**: equity markets down 30%
- **Rate shock**: parallel shift up 200bps
- **Credit widening**: spreads widen 150bps
- **FX stress**: USD and GBP depreciate 15% vs EUR
- **Combined**: simultaneous equity, rate, credit and FX shock
- **Historical**: 2008 financial crisis, 2020 COVID crash, 2022 rate shock

> **Methodology note**: in this project, stress P&L uses first-order sensitivities (delta for
> equities, modified duration for rates and credit, direct revaluation for FX).
> In production these figures come from a third-party risk system or modeled to higher orders.

In [ ]:
from src.risk.risk_utils import HISTORICAL_SCENARIOS
phtml.display_historical_scenarios(HISTORICAL_SCENARIOS)


In [ ]:

custom_scenarios = {
    'Equity Crash -30%'      : rk.stress_equity(risk_df, delta_equity=-0.30),
    'Rate Shock +200bps'     : rk.stress_rates(risk_df, delta_y=0.02),
    'Credit Widening +150bps': rk.stress_credit(risk_df, delta_spread=0.015),
    'FX Stress USD/GBP -15%' : rk.stress_fx(risk_df, fx_shocks={'USD': -0.15, 'GBP': -0.15}),
    'Combined'               : rk.stress_combined(risk_df),
}
    
phtml.display_scenarios(risk_df, custom_scenarios, add_historical=True)

---
## 6. Liquidity Profile

### 6.1 Buckets


ESMA guidelines (ESMA34-39-897) require AIFMs to categorise portfolio assets
by time to liquidation under normal market conditions. Results are reported
to the CSSF via Annex IV and used to assess redemption coverage.

Liquidity buckets (ESMA standard):

<small>

| Bucket | Time to liquidate |
|--------|------------------|
| 1      | 1 day            |
| 2      | 2-7 days         |
| 3      | 8-30 days        |
| 4      | 31-90 days       |
| 5      | 91-365 days      |
| 6      | > 1 year         |

</small>


ESMA buckets are roughly: day, week, month, quarter, year, or above.

Days to liquidate per position:

$$\text{days}_i = \frac{|MV_i|}{ADV_i \times \text{pct\_adv}}$$

* **ADV** (Average Daily Volume): average contracts/shares traded per day over
a 20-day window, sourced from Bloomberg. 
* **pct_adv = 25%** is an internal
RMP parameter representing the maximum fraction of ADV tradeable per day
without significant market impact. 
* Direct properties and instruments with
zero ADV are classified as illiquid (> 1 year).
* Cash and money market instruments are classified as 1 day by definition,
as they require no liquidation process.

In [ ]:
# MRS-30: Liquidity profile
liq = rk.compute_liquidity_profile(risk_df, NAV, pct_adv=0.25)
risk_df_liq = liq['risk_df_liq']
bucket_full = liq['bucket_full']

phtml.display_buckets(bucket_full, risk_df_liq, NAV)

In [ ]:
from src.ui.liquidity_plot import plot_liquidity_profile
plot_liquidity_profile(bucket_full, FUND_ID, metric='pct_nav_abs')


---
### 6.2 Redemption Stress

AIFMD Article 16 and ESMA/2020/1498 require AIFMs to assess whether the
fund can meet redemptions under normal and stressed conditions. Assets
liquidatable within the contractual notice period (buckets `'1 day'` and
`'2-7 days'`) are compared against the redemption amount. A shortfall
triggers a gate or side-pocket recommendation to the risk committee.

<small>

| Scenario | Redemption |  Rationale |
| --- | --- | --- |
| Normal | 10% NAV | Routine investor exit |
| Large | 25% NAV | Large single redemption |
| Stress | 50% NAV | Co-ordinated stress exit |
| Largest investor | Fund-specific | Concentration stress |

In [ ]:
REDEMPTION_SCENARIOS = [
    (0.10, 'Normal'),
    (0.25, 'Large'),
    (0.50, 'Stress'),
]
phtml.display_redemption_stress(FUND_ID, NOTICE, REDEMPTION_SCENARIOS, NAV, risk_df_liq)


### 6.3 Investor Concentration

**ESMA thresholds** (ESMA/2020/1498, Annex VI):
- Single investor **> 20% of NAV** → concentration risk flag
- Top 3 investors **> 50% of NAV** → high-concentration flag

A concentrated investor base amplifies redemption risk: one large exit
can force asset sales that affect all remaining investors. The largest-investor
scenario below connects MRS-31 and MRS-32 — it uses the investor register to
derive the fourth redemption stress scenario.

In [ ]:
# MRS-32: Investor concentration — AIFM Hedge Fund
_investors = rk.load_investor_register(FUND_ID, NAV)

_conc = rk.investor_concentration(_investors, NAV, threshold=0.20)
_top  = _conc['by_investor']
_type = _investors.set_index('investor_id')['investor_type']


In [ ]:
phtml.display_inv_concentration(NAV, risk_df_liq, _investors, _conc, _top, _type)

### 6.4 Counterparty Stress

AIFMD Annex VI requires AIFMs to stress-test **counterparty default** scenarios. For a long/short equity and credit strategy the key counterparties are the prime broker(s) and OTC derivatives (ISDA) counterparties. The relevant exposure is the **net uncollateralised** position after applying initial margin and variation margin held at the counterparty.

> Collateral coverage is simulated. A production system would derive these figures from the daily collateral reconciliation.


In [ ]:
# MRS-35: Counterparty stress — AIFM Hedge Fund
# Simulated prime brokerage and ISDA OTC derivatives counterparty register
# _cp_hf = rk.load_counterparty(FUND_ID) 
counterparty_stress = rk.compute_counterparty_stress(FUND_ID, ENGINE, NAV)
phtml.display_counterparty_stress(NAV, **counterparty_stress)

### 6.5 Combined Stress (Market + Liquidity)

ESMA/2020/1498 Annex VI requires a **combined stress** scenario: what happens if the market falls and investors simultaneously redeem? For this fund the combined shock is **equity −20%** (market stress) **and** **25% NAV redemption** (liquidity stress) applied simultaneously.

Under combined stress, liquid assets also decline in value — equity held as collateral or as liquidatable positions shrinks by 20%. The redemption demand is computed on the **pre-stress NAV** (investor expectation at point of submission).


In [ ]:
# MRS-35: Combined stress — equity -20% AND 25% redemption simultaneously
phtml.display_combined_stress_mkt_plus_liq(
    risk_df=risk_df,
    risk_df_liq=risk_df_liq,
    nav=NAV,
    notice_days=NOTICE,
    delta_equity=-0.20,
    redemption_pct=0.25,
);


#### 6.6. A note on reconciling exposures
<small>

| Measure | Numerator | Cash | Shorts | Derivatives |
|---|---|---|---|---|
| NAV | signed market value | included | netted | market value | 
| Liquidity buckets | abs(market value) | included | abs | abs(market value) | 
| Gross leverage | abs(market value) / notional | excluded | included | full notional | 
| Commitment leverage | net netting | excluded | netted | delta-adjusted notional | 

</small>

- **NAV** = sum of signed market values including cash. Shorts reduce NAV.
- **Liquidity bucket total** = sum of abs(market value) including cash. Always >= NAV due to short positions contributing positively.
- **Gross leverage numerator** = abs exposures excluding cash + derivative full notional. Excludes cash so will not match NAV.
- **Commitment leverage numerator** = uses delta-adjusted derivative notional and nets hedging positions. Excludes cash. As deltas are usually bellow 1, hedges are netted typically is less than gross leverage.

Conclusion: Gross and commitment are regulatory leverage metrics with specific inclusion/exclusion rules defined in EU 231/2013. They are not designed to reconcile to NAV; they measure economic risk exposure, not portfolio value.

## 7. P&L Attribution by Risk Factor

Under AIFMD Article 15, the risk function must be able to explain where
returns and losses come from — not just report the total NAV move. The CSSF
expects the risk manager to answer "what drove the loss on day X" by factor.

This section uses **sensitivity-based attribution**. Regression-based
approaches give average historical loadings and cannot reflect current
position changes. Sensitivity-based attribution uses actual positions and
actual market moves each day, consistent with how VaR is computed.

> This is a risk governance output, not a direct Annex IV or Annex VI field.
> It feeds the Board risk report and supports the AIFMD Article 15 evidence pack.

### Attribution framework

#### A. Total

$\text{Total P\&L} = \text{Equity P\&L} + \text{Rates P\&L} + \text{FX P\&L} + \text{Residual}$

---

#### B. Components

$\text{Equity P\&L} = \sum_i \beta_i \times r_{market} \times MV_i$

$\text{Rates P\&L} = \sum_i -D_i \times \Delta y \times MV_i$

$\text{FX P\&L} = \sum_i \text{notional}_{foreign,i} \times r_{FX,i}$

where $\beta_i$ is the 1-year rolling beta to the benchmark, $D_i$ is
modified duration, and $r_{FX,i}$ is the daily FX return vs EUR.

---

#### C. Residual

$\text{Residual} = \text{P\&L}_{actual} - (\text{Equity} + \text{Rates} + \text{FX})$

A large or persistent residual signals model limitations, missing factors
(credit spread, volatility, carry), or data issues. It is shown, not suppressed.

**% explained** $= |\text{explained}| / |\text{actual}|$. Target $\geq 80\%$.

In [ ]:
from src.risk.pnl_attribution import compute_daily_attribution
from src.ui.attribution_plot import plot_attribution_cumsum

# Compute attribution
attr_result = compute_daily_attribution(
    engine=ENGINE,
    fund_id=FUND_ID,
    bbg=BBG,
    valuation_date=VALUATION_DATE,
    residual_threshold_pct=0.20,
)

attr = attr_result['attr']
flagged = attr_result['flagged']
attr_cumsum = attr_result['attr_cumsum']

# Plot
plot_attribution_cumsum(attr_cumsum, FUND_ID)

# Model quality report
prt.print_attribution(attr, flagged)


**Methodology and limitations**

Sensitivity-based attribution using actual daily positions and market moves.
Regression-based attribution is not used — it gives average historical loadings
and cannot reflect current position composition. 

Benchmark: SPY (S&P 500) — appropriate for this USD-heavy long/short equity portfolio.

**Limitations:**
* Single-factor equity model — no sector, style, or stock-level decomposition
* Rate attribution uses a simulated parallel shift — no key-rate DV01
* FX covers EUR/USD only
* Position composition is static in this simulation

**Regulatory context:** this section is an internal governance output consumed
by the Board and risk committee. It supports the AIFMD Article 15 evidence
pack and CSSF expectations for risk management reporting. It is not a direct
Annex IV or Annex VI deliverable.

---

## 8. Pre-Trade Compliance Check

The pre-trade check (`pre_trade_check` in `risk_utils.py`) models AIFMD Article 15(3)(c): the risk management function must be able to assess the impact of any proposed investment on the fund's risk profile **before** execution. It is run as a blocking gate — a trade flagging a breach is escalated to the PM and CRO before order placement.

`pre_trade_check` (in `risk_utils.py`) works in three steps:
1. Loads the current enriched portfolio via `get_risk_ready_df`
2. Constructs a pro-forma position set after applying the proposed trade (`_ptc_apply_trade`)
3. Runs AIFM Hedge Fund checks: gross leverage, commitment leverage, single-issuer \
concentration, sector concentration, counterparty exposure (OTC), short-selling \
reportability (EU 236/2012), and liquidity impact

### Checks performed

<small>

| # | Check | Limit | Basis |
|---|---|---|---|
| 1 | Gross leverage | 300% NAV | EU 231/2013 Art. 7; Recitals 13–14 |
| 2 | Commitment leverage | 200% NAV | EU 231/2013 Art. 8 |
| 3 | Single-issuer concentration | 25% NAV | Internal RMP |
| 4 | Sector concentration (GICS, corporate only) | 30% NAV | Internal RMP |
| 5 | Counterparty exposure (OTC derivatives) | 10% / 5% NAV | EU 231/2013 Art. 43 |
| 6 | Net short position | 0.2% NAV reportable | EU 236/2012 Art. 5 |
| 7 | Liquidity — days to liquidate | Fund-specific | AIFMD Art. 16 |

</small>


### Borrowings — Recital 13 correction

Borrowings are included in **both** gross and commitment exposure at absolute value (EU 231/2013 Recital 13). The only exclusion is capital call credit facilities that are temporary and fully covered by LP commitments (Recital 14) — not applicable to this hedge fund. The `pct_financed` field on each trade controls the funding source:

- `pct_financed = 0.0` — cash-funded: cash position reduced by the full notional, no new borrowing
- `pct_financed = 1.0` — prime broker financed: a `Borrowing` row is created in the pro-forma and counted in both leverage methods

### Pre-existing breaches

Checks [3] and [6] only flag a breach if the **proposed trade makes the breaches to get worse**. 


### 8.1  Scenario 1 — Clean buy (passes all checks)

New investment-grade bond, issuer not currently in the portfolio. \
EUR 3M notional on a fund NAV of approximately EUR 94.5M (≈ 3.2% of NAV). \
Post-trade gross leverage stays well below 300%; new issuer is below the 25% RMP concentration \
limit; eligible instrument. **Expected result: PASS.**

In [ ]:
# define once at setup (same cell as ENGINE, BBG, etc.)
PTC_CTX = dict(
    engine            = ENGINE,
    fund_id           = FUND_ID,
    date              = VALUATION_DATE,
    counterparties_df = _cp_hf,
    bbg               = BBG,
    deriv_bbg_map     = deriv_bbg_map,
    currency_bbg_map  = currency_bbg_map,
)

In [ ]:
# Deutsche Bank IG bond — new name, small size
trade_pass = {
    'isin'           : 'XS2500000001',
    'direction'      : 'buy',
    'quantity'       : 3_000_000,
    'price_eur'      : 1.00,
    'counterparty'   : 'BNP Paribas',      
    'underlying_risk': 'Deutsche Bank AG',     
    'asset_class'    : 'Bond',
    'sub_asset_class': 'IG Corporate',
    'dur_adj_mid'    : 4.2,
    'currency'       : 'EUR',
    'adv_eur'        : 5_000_000,
    'issuer'         : 'Deutsche Bank AG',
    'sector'         : 'Financials',
    'pct_financed'   : 0.0,   # cash-funded IG bond buy

}

result_pass = rk.pre_trade_check(trade_pass, **PTC_CTX)
phtml.display_ptc(result_pass, test_number=1)
               




### 8.2  Scenario 2 — Gross and commitment leverage breach

A large new long equity position: 750,000 shares of Goldman Sachs at EUR 200/share \
= EUR 150M notional. The fund's current gross exposure is approximately EUR 152M on a \
NAV of EUR 94.5M (≈ 1.6×). Adding EUR 150M pushes gross exposure to ≈ EUR 302M = **3.20× NAV**, \
exceeding the 300% RMP limit. Commitment leverage also breaches. \
The single new issuer at EUR 150M would additionally represent ≈ 158% NAV, \
far above the 25% single-issuer concentration limit.

> This scenario deliberately stacks multiple breaches: it shows that `pre_trade_check` \
catches all of them simultaneously and presents each with its limit and actual value.

In [ ]:
# Goldman Sachs — oversized long, breaches gross leverage and issuer concentration
trade_leverage = {
    'isin'           : 'US38141G1040',
    'direction'      : 'buy',
    'quantity'       : 750_000,
    'price_eur'      : 200.00,
    'counterparty'   : 'Deutsche Bank AG',      
    'underlying_risk': 'Goldman Sachs',                 
    'sub_asset_class': 'Large Cap',
    'beta'           : 1.40,
    'currency'       : 'USD',
    'adv_eur'        : 150_000_000,
    'issuer'         : 'Goldman Sachs Group',
    'pct_financed'   : 1.0,   # fully prime-broker financed
}

result_pass = rk.pre_trade_check(trade_leverage, **PTC_CTX)
phtml.display_ptc(result_pass, test_number=2)


### 8.3  Scenario 3 — Single-issuer concentration breach only

 is not currently in the portfolio. Buying 150,000 shares at EUR 170/share \
= EUR 25.5M notional. On NAV of EUR 94.5M, this is **27.0%**, exceeding the 25% RMP \
single-issuer limit. The size is deliberately chosen to breach concentration while \
staying well within the 300% gross leverage limit (post-trade gross ≈ 1.88× NAV).

This is the typical operational question: the portfolio manager wants to open a meaningful \
new position. The risk function must confirm the proposed allocation is within limits before \
the order reaches the market. A minor reduction in size (below EUR 23.6M = 25% NAV) \
would resolve the breach.

In [ ]:
# Alphabet Inc — new position, sized to breach issuer concentration (27% NAV)
trade_conc = {
    'isin'           : 'US02079K3059',
    'direction'      : 'buy',
    'quantity'       : 150_000,           # 150 k × EUR 170 = EUR 25.5 M
    'price_eur'      : 170.00,
    'asset_class'    : 'Equity',
    'sub_asset_class': 'Large Cap',
    'beta'           : 1.15,
    'currency'       : 'USD',
    'adv_eur'        : 200_000_000,
    'issuer'         : 'Alphabet Inc',
    'sector'         : 'Technology',
    'pct_financed'   : 0.0,   # cash-funded equity buy
}

result_conc = rk.pre_trade_check(trade_conc, **PTC_CTX)
phtml.display_ptc(result_conc, test_number=3)


----


## ESG Risk Indicators

ESG monitoring is required under CSSF Regulation 22-05 and SFDR (EU 2019/2088).
Portfolio-level ESG scores are computed as NAV-weighted averages. 

**SFDR classification**: Article 6 (no sustainability claim). ESG factors are
monitored but do not drive investment decisions. Article 8 would require documented
ESG screening; Article 9 would require sustainable investment as the primary objective.

Metrics monitored:
- Weighted average ESG score (composite, E, S, G)
- % of NAV in instruments with ESG score below 30*
- % of NAV with active controversy flag
- Weighted average carbon intensity (tCO2/EURm revenue)

> **Note**: ESG scores for liquid instruments are fetched from Bloomberg at
> enrichment time. Instruments without a Bloomberg ticker use fund administrator
> ESG data embedded in the position file.


> % of NAV in instruments with ESG score below internal threshold. 
> Threshold is defined in the Risk Management Policy. 
> ESG scores are not comparable across providers as each
> uses a different methodology and scale.
> (here using 30/100,Bloomberg scale 0-100 higher is better)

> **Scale note**: all ESG scores in this framework use a 0-100 scale where
> 100 is best, consistent with Bloomberg ESG methodology. Illiquid instrument
> scores are provided by the fund administrator on the same scale. In production,
> the ManCo should ensure all third-party ESG data is normalised to a consistent
> scale before aggregating portfolio-level metrics.

> **ESG look-through for derivatives**: derivatives have indirect ESG exposure
> through their underlying. The delta-adjusted notional is used as the ESG
> exposure weight rather than market value, which would understate the exposure.
>
> For options:
> $$ESG\_exposure_i = |\delta_i| \times Q_i \times \text{contract size} \times P_{underlying} \times FX$$
>
> For futures: full notional is used since delta = 1.
>
> For FX forwards: no ESG exposure assigned (no issuer).

In [ ]:
esg_df  = esg_u.build_esg_df(risk_df, BBG, ENGINE, FUND_ID, VALUATION_DATE)
esg_u.display_esg_assets(esg_df)

In [ ]:
esg_u.display_esg_summary(esg_df)

In [ ]:
esg_u.plot_esg_profile(esg_df, FUND_ID, plot_title='06. ESG profile - HF')

---

## 9. Annex IV Report 
<span style="color:gray"><i>MRS-34</i></span>

AIFMD Article 110 requires the AIFM to submit a quarterly **transparency report**
to the CSSF (Annex IV template). The report covers fund identity, strategy,
risk profile, leverage (gross and commitment method), and liquidity.
ESMA technical guidance v1.7 (July 2024) defines the mandatory field set.

AIFMD II (Directive 2024/927/EU) expanded the leverage disclosure to a granular
breakdown by source: financial borrowing, synthetic leverage through derivatives,
and repo/reverse repo — making it easier for the CSSF to identify funds building
leverage through derivatives rather than direct borrowing.

**Regulatory basis:** AIFMD Art. 110 / EU 231/2013 Annex IV / ESMA v1.7 (Jul 2024)


In [ ]:
from src.reporting.annex_iv import build_annex_iv, export_annex_iv_excel

ANNEX_IV_QUARTER = '2026-03-31'

annex_iv = build_annex_iv(ENGINE, FUND_ID, quarter=ANNEX_IV_QUARTER)
print(f"Annex IV — {FUND_ID}   Reporting period: Q1 2026 ({ANNEX_IV_QUARTER})")
print(f"NAV: EUR {annex_iv['_nav']/1e6:,.1f}M")

In [ ]:
# Build Annex IV data
from src.ui.annex_iv_display import annex_iv_section

ANNEX_IV_QUARTER = '2026-03-31'
annex_iv = build_annex_iv(ENGINE, FUND_ID, quarter=ANNEX_IV_QUARTER)

# ═══════════════════════════════════════════════════════════════════════════════
# DISPLAY: All Annex IV sections
# ═══════════════════════════════════════════════════════════════════════════════

annex_iv_section(annex_iv, 'identification');
annex_iv_section(annex_iv, 'breakdown');
annex_iv_section(annex_iv, 'risk_measures');
annex_iv_section(annex_iv, 'leverage_detail');
annex_iv_section(annex_iv, 'liquidity_buckets');
annex_iv_section(annex_iv, 'liquidity_terms');

# Export to Excel
path = export_annex_iv_excel(ENGINE, quarter=ANNEX_IV_QUARTER)
print(f"Annex IV workbook written: {path}")
